In [25]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("ntphiep/viT5_tst_coarse", use_fast=False)
model = AutoModelForSeq2SeqLM.from_pretrained("ntphiep/viT5_tst_coarse")

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'MT5Tokenizer'. 
The class this function is called from is 'T5Tokenizer'.
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [4]:
import sagemaker
import boto3
from sagemaker.huggingface import HuggingFaceModel

role = "arn:aws:iam::014498663963:role/service-role/AmazonSageMaker-ExecutionRole-20250504T220510"

# Hub Model configuration. https://huggingface.co/models
hub = {
	'HF_MODEL_ID':'ntphiep/viT5_tst_coarse',
	'HF_TASK':'text-generation'
}

# create Hugging Face Model Class
huggingface_model = HuggingFaceModel(
	transformers_version='4.49.0',
	pytorch_version='2.6.0',
	py_version='py312',
	env=hub,
	role=role, 
)

# deploy model to SageMaker Inference
predictor = huggingface_model.deploy(
	initial_instance_count=1, # number of instances
	instance_type='ml.m4.xlarge' # ec2 instance type
)

predictor.predict({
	"inputs": "Can you please let us know more details about your ",
})

------------------------------------------------*

Please check the troubleshooting guide for common errors: https://docs.aws.amazon.com/sagemaker/latest/dg/sagemaker-python-sdk-troubleshooting.html#sagemaker-python-sdk-troubleshooting-create-endpoint


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:23                                                                                   │
│                                                                                                  │
│   20 )                                                                                           │
│   21                                                                                             │
│   22 # deploy model to SageMaker Inference                                                       │
│ ❱ 23 predictor = huggingface_model.deploy(                                                       │
│   24 │   initial_instance_count=1, # number of instances                                         │
│   25 │   instance_type='ml.m4.xlarge' # ec2 instance type                                        │
│   26 )                                                                                           │
│                                                                                                  │
│ C:\Users\Hiep\AppData\Roaming\Python\Python312\site-packages\sagemaker\huggingface\model.py:326  │
│ in deploy                                                                                        │
│                                                                                                  │
│   323 │   │   │   │   inference_tool=inference_tool,                                             │
│   324 │   │   │   )                                                                              │
│   325 │   │                                                                                      │
│ ❱ 326 │   │   return super(HuggingFaceModel, self).deploy(                                       │
│   327 │   │   │   initial_instance_count,                                                        │
│   328 │   │   │   instance_type,                                                                 │
│   329 │   │   │   serializer,                                                                    │
│                                                                                                  │
│ C:\Users\Hiep\AppData\Roaming\Python\Python312\site-packages\sagemaker\model.py:1814 in deploy   │
│                                                                                                  │
│   1811 │   │   │   │   )                                                                         │
│   1812 │   │   │   │   self.sagemaker_session.update_endpoint(self.endpoint_name, endpoint_conf  │
│   1813 │   │   │   else:                                                                         │
│ ❱ 1814 │   │   │   │   self.sagemaker_session.endpoint_from_production_variants(                 │
│   1815 │   │   │   │   │   name=self.endpoint_name,                                              │
│   1816 │   │   │   │   │   production_variants=[production_variant],                             │
│   1817 │   │   │   │   │   tags=tags,                                                            │
│                                                                                                  │
│ C:\Users\Hiep\AppData\Roaming\Python\Python312\site-packages\sagemaker\session.py:6033 in        │
│ endpoint_from_production_variants                                                                │
│                                                                                                  │
│   6030 │   │   logger.info("Creating endpoint-config with name %s", name)                        │
│   6031 │   │   self.sagemaker_client.create_endpoint_config(**config_options)                    │
│   6032 │   │                                                                                     │
│ ❱ 6033 │   │   return self.create_endpoint(                                                      │
│   6034 │   │   │   endpoint_name=name,                     

In [13]:
from transformers import MT5Tokenizer, AutoModelForSeq2SeqLM

tokenizer = MT5Tokenizer.from_pretrained("ntphiep/viT5_tst_formal")
model = AutoModelForSeq2SeqLM.from_pretrained("ntphiep/viT5_tst_formal")

def predict(text):
    inputs = tokenizer(text, return_tensors="pt", padding='longest', max_length=64)
    input_ids = inputs.input_ids
    attention_mask = inputs.attention_mask
    output = model.generate(input_ids, attention_mask=attention_mask, max_length=256, top_p=0.95)
    return tokenizer.decode(output[0], skip_special_tokens=True)


text = "Bọn công nhân thì được trả bằng thóc, một thằng bình thường kiếm được có 5 bao rưỡi thóc một tháng, còn thằng quản đốc thì được tận 7 bao rưỡi."
result = predict(text) 
print("👉 Output:", result)


The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'MT5Tokenizer'. 
The class this function is called from is 'T5Tokenizer'.
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


👉 Output: Công nhân được trả lương bằng thóc, một người bình thường thu được 5 bao rưỡi thóc mỗi tháng, trong khi người quản đốc được hưởng 7 bao nhiêu?


In [5]:
CKPT = 'ntphiep/viT5_tst_coarse'
from transformers import MT5Tokenizer, MT5ForConditionalGeneration
tokenizer = MT5Tokenizer.from_pretrained(CKPT)
model = MT5ForConditionalGeneration.from_pretrained(CKPT)

def paraphase(text):
    inputs = tokenizer(text, return_tensors='pt')
    input_ids = inputs.input_ids
    attention_mask = inputs.attention_mask
    output = model.generate(input_ids, attention_mask=attention_mask, max_length=64)
    return tokenizer.decode(output[0], skip_special_tokens=True)

text = "Cục Quản lý Dược, Bộ Y tế vừa ra quyết định đình chỉ lưu hành, thu hồi và yêu cầu tiêu hủy toàn quốc đối với lô sản phẩm sữa rửa mặt Gammaphil - chai 125ml do phát hiện chứa các chất không nằm trong công thức đã được công bố."
        

print(paraphase(text))


C:\Users\Hiep\AppData\Roaming\Python\Python312\site-packages\huggingface_hub\file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'MT5Tokenizer'. 
The class this function is called from is 'T5Tokenizer'.
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Cục Quản lý Dược, Bộ Y tế vừa ra lệnh cấm mẹ nó lưu hành, thu hồi với yêu cầu tiêu hủy hết mẹ cái lô sữa rửa mặt Gammaphil - chai 125ml vì tội không đúng công thức.


In [1]:
### REAL

from sagemaker.huggingface import HuggingFaceModel
import sagemaker

role = "arn:aws:iam::014498663963:role/service-role/AmazonSageMaker-ExecutionRole-20250504T220510"
sess = sagemaker.Session()

huggingface_model = HuggingFaceModel(
    model_data="s3://hiep-delta-bk/models/coarse_v1.tar.gz",
    role=role,
    transformers_version="4.49.0",  # hoặc version m training
    pytorch_version="2.6.0",
    py_version="py312",
    env={
        "HF_TASK": "text2text-generation"  # Hoặc text-classification, fill-mask, question-answering,...
    }
)


predictor = huggingface_model.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.xlarge",  # Nếu model nhỏ, CPU ok. Muốn nhanh thì "ml.g4dn.xlarge" (GPU)
)


sagemaker.config INFO - Not applying SDK defaults from location: C:\ProgramData\sagemaker\sagemaker\config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: C:\Users\Hiep\AppData\Local\sagemaker\sagemaker\config.yaml


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:21                                                                                   │
│                                                                                                  │
│   18 )                                                                                           │
│   19                                                                                             │
│   20                                                                                             │
│ ❱ 21 predictor = huggingface_model.deploy(                                                       │
│   22 │   initial_instance_count=1,                                                               │
│   23 │   instance_type="ml.m5.xlarge",  # Nếu model nhỏ, CPU ok. Muốn nhanh thì "ml.g4dn.xlar    │
│   24 )                                                                                           │
│                                                                                                  │
│ C:\Users\Hiep\AppData\Roaming\Python\Python312\site-packages\sagemaker\huggingface\model.py:326  │
│ in deploy                                                                                        │
│                                                                                                  │
│   323 │   │   │   │   inference_tool=inference_tool,                                             │
│   324 │   │   │   )                                                                              │
│   325 │   │                                                                                      │
│ ❱ 326 │   │   return super(HuggingFaceModel, self).deploy(                                       │
│   327 │   │   │   initial_instance_count,                                                        │
│   328 │   │   │   instance_type,                                                                 │
│   329 │   │   │   serializer,                                                                    │
│                                                                                                  │
│ C:\Users\Hiep\AppData\Roaming\Python\Python312\site-packages\sagemaker\model.py:1737 in deploy   │
│                                                                                                  │
│   1734 │   │   │   return None                                                                   │
│   1735 │   │                                                                                     │
│   1736 │   │   else:  # existing single model endpoint path                                      │
│ ❱ 1737 │   │   │   self._create_sagemaker_model(                                                 │
│   1738 │   │   │   │   instance_type=instance_type,                                              │
│   1739 │   │   │   │   accelerator_type=accelerator_type,                                        │
│   1740 │   │   │   │   tags=tags,                                                                │
│                                                                                                  │
│ C:\Users\Hiep\AppData\Roaming\Python\Python312\site-packages\sagemaker\model.py:986 in           │
│ _create_sagemaker_model                                                                          │
│                                                                                                  │
│    983 │   │   │   │   enable_network_isolation=self._enable_network_isolation,                  │
│    984 │   │   │   │   tags=format_tags(tags),                                                   │
│    985 │   │   │   )                                                                             │
│ ❱  986 │   │   │   self.sagemaker_session.create_model(**create_model_args)                      │
│    987 │                                                   